## 1. Configuración Inicial y Carga de Datos

En esta primera sección importamos todas las herramientas necesarias (traductor, detector de idiomas y utilidades de paralelización). A continuación, cargamos el dataset df_comentarios_limpio.csv generado en el paso anterior y aplicamos una pequeña limpieza de valores nulos o vacíos para que no interfieran en el proceso de traducción.

In [9]:
# Instalación de librerías (descomentar si es necesario)
#!pip install deep-translator langdetect -q

# Librerías
import pandas as pd
import os
import time
from langdetect import detect
from langdetect.detector_factory import LangDetectException
from deep_translator import GoogleTranslator
from concurrent.futures import ThreadPoolExecutor, as_completed

# Función para limpiar nulos o cadenas vacías antes de traducir
def gestionar_nulos(df, columnas):
    for col in columnas:
        df[col] = df[col].fillna('')
    # Nos quedamos con las filas que NO tengan todas las columnas de interés vacías
    df = df[~df[columnas].apply(lambda row: all(v == '' for v in row), axis=1)]
    return df

# Carga de datos
ruta_entrada = '..datos/procesados/df_comentarios_limpio.csv'
df_comentarios_limpio = pd.read_csv(ruta_entrada, encoding='utf-8-sig')

# Limpieza de nulos inicial
df_comentarios_limpio = gestionar_nulos(df_comentarios_limpio, df_comentarios_limpio.columns.tolist())
print(f"Dimensiones del dataset listo para traducir: {df_comentarios_limpio.shape}")
display(df_comentarios_limpio.head(3))

Dimensiones del dataset listo para traducir: (58720, 11)


,nombre_del_hotel,nacionalidad,detalles_hab,noches,personas,comentario_general,positivo,negativo,nota,fecha,comentario_media_hotel
0,barceló valencia,colombia,,2,familia,espectacular,espectacular todo! restaurante; servicio; habi...,todo me gustó,10.0,2026-04-14,fabuloso
1,barceló valencia,españa,,1,pareja,muy bien,,,9.0,2026-04-05,fabuloso
2,barceló valencia,españa,2 habitaciones dobles deluxe comunicadas,1,familia,fantástico,"habitaciones amplias, limpieza y ubicación",,9.0,2026-04-03,fabuloso


## 2. Motores de Detección y Traducción en Lote

Aquí definimos las dos funciones principales (los motores) de nuestra traducción. La primera (detectar_idioma) analiza cada texto para saber si ya está en español (y así ahorrar tiempo y peticiones). La segunda (traducir_lote) se encarga de enviar paquetes de textos a la API de Google Translate, incorporando un sistema de reintentos automáticos (con pausas) por si el servidor bloquea las peticiones temporalmente.

In [10]:
def detectar_idioma(texto):
    """Detecta el idioma de un texto. Si está vacío o falla, asume 'es' o 'desconocido'."""
    try:
        if pd.isna(texto) or str(texto).strip() == '':
            return 'es'
        return detect(str(texto))
    except LangDetectException:
        return 'desconocido'

def traducir_lote(textos):
    """Traduce una lista de textos con reintentos y esperas más largas."""
    intentos = 3
    translator = GoogleTranslator(source='auto', target='es')

    for intento in range(intentos):
        try:
            # IMPORTANTE: translate_batch en deep-translator NO es una sola petición atómica
            return translator.translate_batch(textos)
        except Exception as e:
            print(f"Error en lote (intento {intento+1}/{intentos}): {e}")
            if "too many requests" in str(e).lower():
                # Si nos banean, esperamos mucho más (3, 6, 9 minutos)
                espera = 180 * (intento + 1)
                print(f"  Rate limit alcanzado. Esperando {espera}s para enfriar IP...")
                time.sleep(espera)
            elif intento < intentos - 1:
                espera = 60 * (intento + 1)
                time.sleep(espera)

    return textos # Mantiene original si falla todo

## 3. Orquestador de Traducción con Checkpoints

Esta es la función principal que orquesta el trabajo. Realiza tres tareas clave:

1. Filtra solo los textos que NO están en español.

2. Comprueba si existe un "checkpoint" previo (por si el proceso se interrumpió y queremos retomarlo por donde iba).

3. Envía los textos a traducir usando hilos paralelos (ThreadPoolExecutor) para acelerar el proceso, guardando el progreso cada cierto número de lotes.


In [14]:
import os
import pandas as pd
import time

def traduccion(df, nombre_columna, tamaño_lote=10, delay_between_batches=15):
    """
    Traduce una columna específica y guarda un checkpoint único para esa columna.
    """
    checkpoint_path = f'checkpoint_{nombre_columna}.csv'
    columna = df[nombre_columna].astype(str)

    print(f"\n--- Iniciando traducción de: {nombre_columna} ---")

    # 1. Detección de idiomas
    print("  Detectando idiomas actuales...")
    idiomas = columna.apply(detectar_idioma)
    mask_necesita_traduccion = (idiomas != 'es') & (columna.str.strip() != "")

    indices_totales_a_traducir = columna[mask_necesita_traduccion].index
    print(f"  Total filas en columna: {len(columna)}")
    print(f"  Filas que requieren traducción: {len(indices_totales_a_traducir)}")

    # 2. Cargar progreso previo si existe
    traducciones_hechas = {}
    if os.path.exists(checkpoint_path):
        try:
            # Leemos con utf-8-sig para que los acentos no se rompan
            df_ckpt = pd.read_csv(checkpoint_path, index_col=0, encoding='utf-8-sig')
            # Convertimos a diccionario {indice: texto_traducido}
            traducciones_hechas = df_ckpt.iloc[:, 0].to_dict()
            print(f"  Checkpoint detectado: {len(traducciones_hechas)} traducciones recuperadas.")
        except Exception as e:
            print(f"  No se pudo cargar el checkpoint: {e}")

    # Filtrar los que ya están hechos para obtener solo los pendientes
    indices_pendientes = indices_totales_a_traducir.difference(pd.Index(traducciones_hechas.keys()))
    textos_pendientes = columna.loc[indices_pendientes].tolist()

    if len(textos_pendientes) == 0:
        print("  ¡Todo está ya traducido o en español!")
        return pd.Series(traducciones_hechas)

    print(f"  Pendientes por traducir en esta sesión: {len(textos_pendientes)}")

    # 3. Bucle de traducción por lotes
    try:
        for i in range(0, len(textos_pendientes), tamaño_lote):
            lote_textos = textos_pendientes[i : i + tamaño_lote]
            lote_indices = indices_pendientes[i : i + tamaño_lote]

            print(f"    Traduciendo lote {i//tamaño_lote + 1}...")
            traducidos = traducir_lote(lote_textos)

            # Guardar en nuestro diccionario de memoria
            for idx, texto_trad in zip(lote_indices, traducidos):
                traducciones_hechas[idx] = texto_trad

            # Guardar checkpoint físico cada lote (más seguro)
            # Guardamos solo los índices que ya tenemos traducidos
            pd.DataFrame.from_dict(traducciones_hechas, orient='index', columns=[nombre_columna]).to_csv(checkpoint_path, encoding='utf-8-sig')

            progreso = ((i + len(lote_textos)) / len(textos_pendientes)) * 100
            print(f"    Progreso: {progreso:.1f}% - Checkpoint actualizado.")

            # Pausa para evitar bloqueos de IP
            time.sleep(delay_between_batches)

    except KeyboardInterrupt:
        print("\n  Traducción pausada por el usuario. El progreso está guardado.")
    except Exception as e:
        print(f"\n  Error inesperado: {e}")

    print(f"--- Fin del proceso para {nombre_columna} ---")
    return pd.Series(traducciones_hechas)

## 4. Ejecución por Columnas

Aplicamos la función orquestadora sobre las tres columnas de interés: comentario general, aspectos positivos y aspectos negativos. Configuramos tiempos de espera y tamaños de lote conservadores para evitar que la API gratuita de Google nos bloquee por exceso de peticiones.

In [ ]:
# 1. Traducción del Comentario General
print("Iniciando traducción de: 'comentario_general'")
traducciones_general = traduccion(
    df_comentarios_limpio,
    nombre_columna='comentario_general',
    tamaño_lote=8,           # Un lote un poco más pequeño es más seguro para Google
    delay_between_batches=12 # Espera 12 segundos entre lotes
)

Iniciando traducción de: 'comentario_general'
  Detectando idiomas...
  Total: 58720 | En español/vacío: 25950 | A traducir: 32770
  Cargando checkpoint previo...
  Ya traducidos previamente: 32796 | Quedan: 105
  Progreso: 95.2% -- Checkpoint guardado
¡Traducción de esta columna completada!


In [ ]:
# 2. Traducción de los Aspectos Positivos
print("\nIniciando traducción de: 'positivo'")
traducciones_positivas = traduccion(
    df_comentarios_limpio,
    nombre_columna='positivo',
    tamaño_lote=8,           # Un lote un poco más pequeño es más seguro para Google
    delay_between_batches=12 # Espera 12 segundos entre lotes
)


Iniciando traducción de: 'positivo'
  Detectando idiomas...
Error en lote (intento 3/3): Server Error: You made too many requests to the server.According to google, you are allowed to make 5 requests per secondand up to 200k requests per day. You can wait and try again later oryou can try the translate_batch function
Error en lote (intento 1/3): Server Error: You made too many requests to the server.According to google, you are allowed to make 5 requests per secondand up to 200k requests per day. You can wait and try again later oryou can try the translate_batch function
  Esperando 60s antes de reintentar...
Error en lote (intento 2/3): Server Error: You made too many requests to the server.According to google, you are allowed to make 5 requests per secondand up to 200k requests per day. You can wait and try again later oryou can try the translate_batch function
  Esperando 120s antes de reintentar...


KeyboardInterrupt: 

In [ ]:
# 3. Traducción de los Aspectos Negativos
print("\nIniciando traducción de: 'negativo'")
traducciones_negativas = traduccion(
    df_comentarios_limpio,
    nombre_columna='negativo',
    tamaño_lote=8,           # Un lote un poco más pequeño es más seguro para Google
    delay_between_batches=12 # Espera 12 segundos entre lotes
)


Iniciando traducción de: 'negativo'

--- Iniciando traducción de: negativo ---
  Detectando idiomas actuales...


## 5. Exportación del Dataset Final Traducido

Finalmente, una vez que todas las columnas de texto han sido estandarizadas al español, guardamos el DataFrame resultante. Este dataset es el que finalmente usaremos para tareas analíticas avanzadas, como análisis de sentimiento o modelado de tópicos.

In [ ]:
# 1. Cargar el original de nuevo para asegurar que está limpio
df_final = pd.read_csv('df_comentarios_limpio.csv', encoding='utf-8-sig')

# 2. Leer los 3 archivos que habéis generado por separado
# Usamos index_col=0 porque el índice es la clave para unir los datos
df_gen = pd.read_csv('checkpoint_comentario_general.csv', index_col=0, encoding='utf-8-sig')
df_pos = pd.read_csv('checkpoint_positivo.csv', index_col=0, encoding='utf-8-sig')
df_neg = pd.read_csv('checkpoint_negativo.csv', index_col=0, encoding='utf-8-sig')

# 3. "Pegar" las traducciones en el DataFrame original
df_final.update(df_gen)
df_final.update(df_pos)
df_final.update(df_neg)

# 4. AHORA SÍ, guardar el archivo definitivo con todo unido
ruta_final = 'datos/procesados/comentarios_traducidos_final.csv'
df_final.to_csv(ruta_final, index=False, encoding='utf-8-sig')

print("¡Dataset unificado con éxito!")